# Mascot Video - AI Mascot Overlay Generator

This notebook runs the Mascot Video API server.

**Features:**
- JoyVASA talking-head mascot generation
- Picture-in-picture overlay
- Video effects (brightness, contrast, saturation, gamma)
- Text overlays

## 1. Mount Drive & Check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
!nvidia-smi

## 2. Setup JoyVASA Environment (Part 1)
Installs system packages, PyTorch, Xformers.

**Note:** This may require a runtime restart after completion.

In [ ]:
import os, subprocess

JOYVASA_REPO_PATH = "/content/gdrive/MyDrive/TTDATN/4.Source/JoyVASA"

print("Setting up JoyVASA environment...")

# 1. System packages
print("1. Installing system packages...")
!sudo apt-get install -y ffmpeg git-lfs ninja-build > /dev/null 2>&1
!git lfs install > /dev/null 2>&1

# 2. Clean old libs
print("2. Cleaning old libs...")
!pip uninstall -y torch torchvision torchaudio xformers tensorflow jax jaxlib peft accelerate transformers > /dev/null 2>&1

# 3. Install PyTorch 2.3.1 + Xformers
print("3. Installing PyTorch 2.3.1 & Xformers...")
!pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121
!pip install xformers==0.0.27 --index-url https://download.pytorch.org/whl/cu121

# 4. Numpy & Protobuf
print("4. Downgrading Numpy & Protobuf...")
!pip install "numpy<2" "protobuf==3.20.3"

print("Part 1 done!")

## 3. Setup JoyVASA Environment (Part 2)
Installs JoyVASA requirements, server libs, ONNX, XPose.

**Run this cell after restarting runtime (if needed).**

In [ ]:
import os

JOYVASA_REPO_PATH = "/content/gdrive/MyDrive/TTDATN/4.Source/JoyVASA"

# 5. JoyVASA requirements
req_file = os.path.join(JOYVASA_REPO_PATH, "requirements.txt")
ignore_list = ["torch", "torchvision", "torchaudio", "xformers", "numpy", "protobuf", "onnxruntime-gpu"]

if os.path.exists(req_file):
    with open(req_file, 'r') as f:
        lines = f.readlines()
    clean_req = "requirements_clean.txt"
    with open(clean_req, 'w') as f:
        for line in lines:
            if not any(pkg in line for pkg in ignore_list):
                f.write(line)
    print("5. Installing JoyVASA requirements...")
    !pip install -r {clean_req}

# 6. Server + AI libs
print("6. Installing server & AI libs...")
!pip install transformers==4.41.2 sentence-transformers==3.0.1 accelerate==0.31.0 peft==0.11.1
!pip install fastapi "uvicorn[standard]" python-multipart cloudinary requests pyngrok nest_asyncio qstash

# 7. ONNX Runtime GPU
print("7. Installing ONNX Runtime GPU...")
!pip install onnxruntime-gpu --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/

# 8. Build XPose
print("8. Building XPose...")
xpose_path = os.path.join(JOYVASA_REPO_PATH, "src/utils/dependencies/XPose/models/UniPose/ops")
if os.path.exists(xpose_path):
    os.chdir(xpose_path)
    !pip install .
    os.chdir("/content")
else:
    print("XPose folder not found, skipping")

# 9. bitsandbytes + triton
!pip install "bitsandbytes==0.43.1" "triton==2.3.1"

print("\nSetup complete! Ready to run server.")

## 4. Write Server Code (main.py)

In [ ]:
%%writefile main.py
# --- IMPORTS ---
import os, time, uuid, shutil, re, subprocess, requests
from typing import Dict, Any, Optional
import json

import torch
import cloudinary
import cloudinary.uploader

from fastapi import FastAPI, UploadFile, File, BackgroundTasks, HTTPException, Form
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from qstash import QStash

# --- CONFIG ---
QSTASH_TOKEN = "eyJVc2VySUQiOiJmNTI4OTUyZi00N2NmLTQ0NmMtOTc4OS1jNmI5MGFhODE4OWQiLCJQYXNzd29yZCI6IjYzMDRlYzJjYWExYzRhNWZiMjAxNmQzNjU2ZjdkYjIwIn0="
qstash_client = QStash(QSTASH_TOKEN)

cloudinary.config(
    cloud_name="dbwqzrbur",
    api_key="572214851492358",
    api_secret="0DTsa51ETUU9sK9IPFjoBMP35VI",
    secure=True,
)

OUTPUT_DIR = "outputs"
JOYVASA_REPO_PATH = "/content/gdrive/MyDrive/TTDATN/4.Source/JoyVASA"

# =============================================================================
# FASTAPI APP
# =============================================================================

app = FastAPI(title="AI Mascot Video Generator")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)

jobs: Dict[str, Dict[str, Any]] = {}

# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def clean_filename(filename: Optional[str], fallback: str = "video.mp4") -> str:
    raw = (filename or fallback).strip().replace("\\", "/").split("/")[-1]
    return raw or fallback

def get_original_basename(filename: Optional[str], fallback: str = "video") -> str:
    return os.path.splitext(clean_filename(filename, f"{fallback}.mp4"))[0].strip() or fallback

def strip_highlight_suffix(name: Optional[str], fallback: str = "video") -> str:
    base = get_original_basename(name, fallback)
    if base.lower().endswith("_highlight"):
        base = base[:-len("_highlight")].strip()
    return base or fallback

def upload_to_cloudinary(local_file_path, user_id, folder_name="mascot_videos", job_id="", file_type="mascot", source_original_filename=""):
    print(f"  Uploading to Cloudinary: {local_file_path}...")
    try:
        original_base = get_original_basename(source_original_filename or os.path.basename(local_file_path))
        resp = cloudinary.uploader.upload(
            local_file_path, resource_type="video", folder=folder_name,
            public_id=f"{file_type}_{job_id}" if job_id else os.path.splitext(os.path.basename(local_file_path))[0],
            filename_override=original_base,
            context={"user_id": user_id, "type": file_type, "job_id": job_id},
        )
        url = resp.get("secure_url")
        print(f"  Cloudinary OK: {url}")
        return url
    except Exception as e:
        print(f"  Cloudinary error: {e}")
        return ""

def download_video_from_url(public_url, local_save_path):
    print(f"  Downloading video: {public_url}")
    resp = requests.get(public_url, stream=True, timeout=600)
    resp.raise_for_status()
    with open(local_save_path, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"  Downloaded: {local_save_path}")

def download_image_from_url(url, local_path):
    print(f"  Downloading image: {url}")
    resp = requests.get(url, stream=True, timeout=120)
    resp.raise_for_status()
    with open(local_path, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)

def run_cmd(cmd, check=True, cwd=None):
    print(f"  CMD: {' '.join(cmd)}")
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, cwd=cwd)
    if res.returncode != 0:
        print(f"  STDERR: {res.stderr}")
        if check:
            raise RuntimeError(f"Command failed: {' '.join(cmd)}\n{res.stderr}")
    return res

# =============================================================================
# JOYVASA - MASCOT VIDEO GENERATION
# =============================================================================

def create_mascot_video(job_id, video_path, mascot_image_path, audio_path):
    """Run JoyVASA inference to create a talking mascot video from image + audio.
    Returns the path to the generated mascot video file.
    """
    output_dir = os.path.join(OUTPUT_DIR, f"mascot_{job_id}")
    os.makedirs(output_dir, exist_ok=True)
    output_filename = f"mascot_raw_{job_id}.mp4"
    output_path = os.path.join(output_dir, output_filename)

    # Run JoyVASA inference
    cmd = [
        "python", "inference.py",
        "--source_image", os.path.abspath(mascot_image_path),
        "--driving_audio", os.path.abspath(audio_path),
        "--result_dir", os.path.abspath(output_dir),
    ]
    run_cmd(cmd, cwd=JOYVASA_REPO_PATH)

    # JoyVASA may output with a different name, find the result
    result_files = [f for f in os.listdir(output_dir) if f.endswith(".mp4")]
    if result_files:
        actual_output = os.path.join(output_dir, result_files[0])
        if actual_output != output_path:
            shutil.move(actual_output, output_path)
    else:
        raise RuntimeError(f"JoyVASA produced no output in {output_dir}")

    print(f"  Mascot video created: {output_path}")
    return output_filename

# =============================================================================
# FFMPEG VIDEO PROCESSING
# =============================================================================

def picture_in_picture(main_video, sub_video, output_video, position="bottom-right", margin_x=40, margin_y=40, scale=0.25):
    """Overlay sub_video (mascot) on main_video using ffmpeg PiP."""
    positions = {
        "bottom-right": f"W-w-{margin_x}:H-h-{margin_y}",
        "bottom-left":  f"{margin_x}:H-h-{margin_y}",
        "top-right":    f"W-w-{margin_x}:{margin_y}",
        "top-left":     f"{margin_x}:{margin_y}",
        "center":       "(W-w)/2:(H-h)/2",
    }
    overlay_pos = positions.get(position, positions["bottom-right"])

    cmd = [
        "ffmpeg", "-y",
        "-i", main_video,
        "-i", sub_video,
        "-filter_complex",
        f"[1:v]scale=iw*{scale}:ih*{scale}[pip];[0:v][pip]overlay={overlay_pos}:shortest=1",
        "-c:v", "libx264", "-preset", "ultrafast",
        "-c:a", "aac",
        output_video,
    ]
    run_cmd(cmd)

def process_video_effects(input_path, output_path, brightness=0.0, contrast=1.0, saturation=1.0, gamma=1.0, text_overlays=None):
    """Apply color adjustments and text overlays using ffmpeg."""
    vf_parts = []

    # Color adjustments via eq filter
    eq_parts = []
    if brightness != 0.0:
        eq_parts.append(f"brightness={brightness}")
    if contrast != 1.0:
        eq_parts.append(f"contrast={contrast}")
    if saturation != 1.0:
        eq_parts.append(f"saturation={saturation}")
    if gamma != 1.0:
        eq_parts.append(f"gamma={gamma}")
    if eq_parts:
        vf_parts.append(f"eq={':'.join(eq_parts)}")

    # Text overlays via drawtext filter
    if text_overlays and isinstance(text_overlays, list):
        for overlay in text_overlays:
            text = overlay.get("text", "").replace("'", "'\\''").replace(":", "\\:")
            x = overlay.get("x", 10)
            y = overlay.get("y", 10)
            fontsize = overlay.get("fontsize", 24)
            fontcolor = overlay.get("fontcolor", "white")
            vf_parts.append(
                f"drawtext=text='{text}':x={x}:y={y}:fontsize={fontsize}:fontcolor={fontcolor}"
            )

    if not vf_parts:
        shutil.copy2(input_path, output_path)
        return

    cmd = [
        "ffmpeg", "-y",
        "-i", input_path,
        "-vf", ",".join(vf_parts),
        "-c:v", "libx264", "-preset", "ultrafast",
        "-c:a", "copy",
        output_path,
    ]
    run_cmd(cmd)

# =============================================================================
# BACKGROUND TASK
# =============================================================================

def process_mascot_in_background(
    job_id, user_id, video_path, mascot_image_path, origin_file_name,
    audio_path=None, position="bottom-right", margin_x=40, margin_y=40, scale=0.25,
    text_overlays_raw="", brightness=None, contrast=None, saturation=None, gamma=None,
):
    source_original_basename = strip_highlight_suffix(origin_file_name)
    output_filename = f"mascot_{job_id}.mp4"
    output_video_path = os.path.join(OUTPUT_DIR, output_filename)
    mascot_video_file = None
    total_stages = 4

    try:
        jobs[job_id]["status"] = "processing"

        # Stage 1: Create mascot talking-head video (if audio provided)
        jobs[job_id]["stage"] = f"1/{total_stages}: Creating mascot video"
        mascot_video_full_path = video_path

        if audio_path:
            mascot_video_file = create_mascot_video(job_id, video_path, mascot_image_path, audio_path)
            mascot_video_full_path = os.path.join(OUTPUT_DIR, f"mascot_{job_id}", mascot_video_file)

        # Stage 2: Overlay mascot on main video
        jobs[job_id]["stage"] = f"2/{total_stages}: Overlaying mascot on video"
        pip_output = os.path.join(OUTPUT_DIR, f"pip_{job_id}.mp4")

        if position == "replace":
            shutil.copy2(mascot_video_full_path, pip_output)
        else:
            picture_in_picture(
                main_video=video_path,
                sub_video=mascot_video_full_path,
                output_video=pip_output,
                position=position,
                margin_x=margin_x,
                margin_y=margin_y,
                scale=scale,
            )

        # Stage 3: Apply effects & text overlays
        jobs[job_id]["stage"] = f"3/{total_stages}: Applying effects"
        text_overlays = []
        if text_overlays_raw:
            try:
                text_overlays = json.loads(text_overlays_raw)
                if not isinstance(text_overlays, list):
                    text_overlays = []
            except Exception:
                text_overlays = []

        has_effects = any([
            brightness is not None,
            contrast is not None,
            saturation is not None,
            gamma is not None,
            len(text_overlays) > 0,
        ])

        if has_effects:
            process_video_effects(
                input_path=pip_output,
                output_path=output_video_path,
                brightness=brightness if brightness is not None else 0.0,
                contrast=contrast if contrast is not None else 1.0,
                saturation=saturation if saturation is not None else 1.0,
                gamma=gamma if gamma is not None else 1.0,
                text_overlays=text_overlays,
            )
            if os.path.exists(pip_output):
                os.remove(pip_output)
        else:
            shutil.move(pip_output, output_video_path)

        # Stage 4: Upload
        jobs[job_id]["stage"] = f"4/{total_stages}: Uploading to cloud"
        cloud_url = upload_to_cloudinary(
            output_video_path, user_id,
            f"jobs/{job_id}/{output_filename}", job_id, "mascot",
            source_original_filename=source_original_basename,
        )

        jobs[job_id]["status"] = "completed"
        jobs[job_id]["result"] = {
            "output_filename": output_filename,
            "download_url": cloud_url,
            "source_original_filename": source_original_basename,
        }
        jobs[job_id]["type"] = "mascot"

        try:
            qstash_client.message.publish_json(
                url_group="ai-results",
                body={"user_id": user_id, "type": "mascot", "video_url": cloud_url},
            )
        except Exception as qe:
            print(f"  QStash push failed: {qe}")

        print(f"[{job_id}] Mascot job completed!")

    except Exception as e:
        print(f"[{job_id}] FAILED: {e}")
        jobs[job_id]["status"] = "failed"
        jobs[job_id]["result"] = {"error": str(e)}
    finally:
        for p in [video_path, mascot_image_path]:
            if p and os.path.exists(p):
                os.remove(p)
        if audio_path and os.path.exists(audio_path):
            os.remove(audio_path)
        mascot_temp_dir = os.path.join(OUTPUT_DIR, f"mascot_{job_id}")
        if os.path.exists(mascot_temp_dir):
            shutil.rmtree(mascot_temp_dir)
        if os.path.exists(output_video_path):
            os.remove(output_video_path)

# =============================================================================
# ROUTES
# =============================================================================

@app.post("/mascot", status_code=202)
async def create_mascot_job(
    background_tasks: BackgroundTasks,
    user_id: str = Form(...),
    video_url: str = Form(...),
    mascot_image_url: str = Form(...),
    origin_file_name: str = Form(...),
    audio: UploadFile = File(None),
    position: str = Form("bottom-right"),
    margin_x: int = Form(40),
    margin_y: int = Form(40),
    scale: float = Form(1.0),
    text_overlays: str = Form(""),
    brightness: Optional[float] = Form(None),
    contrast: Optional[float] = Form(None),
    saturation: Optional[float] = Form(None),
    gamma: Optional[float] = Form(None),
):
    """Create mascot overlay video. Matches inference_service /mascot route."""
    job_id = str(uuid.uuid4())
    source_original_basename = strip_highlight_suffix(origin_file_name)

    if not mascot_image_url:
        raise HTTPException(status_code=400, detail="mascot_image_url is required")
    if not os.path.exists(JOYVASA_REPO_PATH):
        raise HTTPException(status_code=503, detail="JoyVASA not available on this instance")

    # Download video locally
    temp_video = f"temp_video_{job_id}.mp4"
    try:
        download_video_from_url(video_url, temp_video)
    except Exception as e:
        if os.path.exists(temp_video):
            os.remove(temp_video)
        raise HTTPException(status_code=400, detail=f"Video download failed: {e}")

    # Download mascot image
    ext = os.path.splitext(mascot_image_url.split("?")[0])[1] or ".png"
    mascot_image_path = f"temp_mascot_{job_id}{ext}"
    try:
        download_image_from_url(mascot_image_url, mascot_image_path)
    except Exception as e:
        for p in [temp_video, mascot_image_path]:
            if os.path.exists(p):
                os.remove(p)
        raise HTTPException(status_code=400, detail=f"Mascot image download failed: {e}")

    # Save audio if provided
    temp_audio_path = None
    if audio and audio.filename:
        audio_content = await audio.read()
        if audio_content:
            temp_audio_path = f"temp_audio_{job_id}{os.path.splitext(audio.filename)[1]}"
            with open(temp_audio_path, "wb") as f:
                f.write(audio_content)

    jobs[job_id] = {"status": "pending", "stage": "Queued", "result": None, "source_original_filename": source_original_basename}
    print(f"[{job_id}] mascot job: pos={position}, scale={scale}, has_audio={temp_audio_path is not None}")

    background_tasks.add_task(
        process_mascot_in_background,
        job_id, user_id, temp_video, mascot_image_path, origin_file_name,
        temp_audio_path, position, margin_x, margin_y, scale,
        text_overlays, brightness, contrast, saturation, gamma,
    )
    return {"job_id": job_id, "status": "pending", "source_original_filename": source_original_basename}


@app.get("/jobs/status/{job_id}")
def get_job_status(job_id: str):
    job = jobs.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail="Job not found")
    return job


@app.get("/download/{job_id}")
def download_video(job_id: str):
    job = jobs.get(job_id)
    if not job or job.get("status") != "completed":
        raise HTTPException(status_code=404, detail="Not ready")
    result = job.get("result", {})
    filename = result.get("output_filename")
    if not filename:
        raise HTTPException(status_code=404, detail="No output file")
    filepath = os.path.join(OUTPUT_DIR, filename)
    if not os.path.exists(filepath):
        raise HTTPException(status_code=404, detail="File not found on disk")
    return FileResponse(path=filepath, media_type="video/mp4")


@app.get("/")
def home():
    return {
        "service": "AI Mascot Video Generator",
        "version": "3.0",
        "joyvasa_path": JOYVASA_REPO_PATH,
        "endpoints": {
            "create_mascot": "POST /mascot (FormData)",
            "job_status": "GET /jobs/status/{job_id}",
            "download": "GET /download/{job_id}",
        },
    }

# =============================================================================
# STARTUP
# =============================================================================

@app.on_event("startup")
async def startup_event():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    app.state.device = DEVICE
    print(f"Device: {DEVICE}")
    print(f"JoyVASA path: {JOYVASA_REPO_PATH}")
    if os.path.exists(JOYVASA_REPO_PATH):
        print("JoyVASA: OK")
    else:
        print("JoyVASA: NOT FOUND - mascot creation will fail")
    print("Mascot server ready!")


## 5. Setup Ngrok

In [ ]:
# Download ngrok binary
!wget -q -c -nc https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar -xzf ngrok-v3-stable-linux-amd64.tgz -C /usr/local/bin
!chmod +x /usr/local/bin/ngrok

## 6. Start Server

In [ ]:
import nest_asyncio
from pyngrok import ngrok, conf
import uvicorn
import asyncio

nest_asyncio.apply()
conf.get_default().ngrok_path = "/usr/local/bin/ngrok"

NGROK_AUTH_TOKEN = "2DebNDD2oRLri75muwiGTPPPpIQ_2K9PEgbE6w1PPpHq62tyA"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(8000)
print(f"Public URL: {public_url}")
print(f"Copy this URL to your inference_service .env COLAB_API_URL")

async def run_server():
    config = uvicorn.Config("main:app", host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    await server.serve()

print("Starting server...")
try:
    await run_server()
except asyncio.CancelledError:
    print("Server stopped.")